In [1]:
import sys
import json
import time
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag.config import CHUNKS_FILE, VECTORS_FILE
from my_rag.chunking import load_model

chunks = []
with open(CHUNKS_FILE, encoding="utf-8") as f:
    for line in f:
        chunks.append(json.loads(line))
vectors = np.load(VECTORS_FILE).astype(np.float32)
model = load_model()
print(len(chunks), vectors.shape)

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 20:49:33.392000 54115 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34979.37it/s]


2848 (2848, 1024)


In [2]:
import re
from rank_bm25 import BM25Okapi

DIACRITICS = re.compile(r"[\u0610-\u061a\u064b-\u065f\u0670\u0640]")
LETTERS = str.maketrans({"أ": "ا", "إ": "ا", "آ": "ا", "ى": "ي", "ة": "ه"})
WORD = re.compile(r"\w+")

def normalize(text):
    text = text.lower()
    text = DIACRITICS.sub("", text)
    return text.translate(LETTERS)

def tokenize(text):
    return WORD.findall(normalize(text))

print(tokenize("إمتى الفُطار؟ What TIME"))
print(tokenize("امتي الفطار"))

corpus = []
for c in chunks:
    corpus.append(tokenize(c["text"]))
bm25 = BM25Okapi(corpus, k1=1.5, b=0.75)

for q in ["ينفع ادخل بدري؟", "رقم الإسعاف 123", "What time is breakfast?"]:
    scores = bm25.get_scores(tokenize(q))
    print(q)
    for i in np.argsort(-scores)[:3]:
        print("   ", round(float(scores[i]), 2), chunks[i]["chunk_id"], "|", chunks[i]["text"][:70].replace("\n", " "))

['امتي', 'الفطار', 'what', 'time']
['امتي', 'الفطار']
ينفع ادخل بدري؟
    16.92 FAQ2561 | Q: ينفع أدخل بدري قبل الميعاد؟ A: الدخول الرسمي الساعة 3 العصر. من 12 
    12.35 FAQ3786 | Q: ينفع أدخل الجيم الساعة 2 بالليل؟ A: الجيم مفتوح 24 ساعة ومجاني لنزل
    11.86 FAQ3609 | Q: من غير بطاقة ينفع أدخل؟ A: لازم كل الكبار يقدموا إثبات شخصية ساري و
رقم الإسعاف 123
    10.9 FAQ3981 | Q: في حالة طوارئ أعمل إيه؟ A: اطلب المساعدة على طول: الإسعاف 123، والش
    10.55 FAQ3978 | Q: ماذا أفعل في حالة طارئة أثناء التحدث مع المساعد؟ A: اطلب المساعدة ف
    8.69 FAQ2309 | Q: هل تمانع في شرح كيفية الاتصال بالشرطة أو الإطفاء أو الإسعاف؟ A: يمك
What time is breakfast?
    12.72 FAQ0755 | Q: What is the standard arrival time? A: Standard check-in time is 3:0
    12.57 FAQ0925 | Q: What is the normal check-in time? A: Standard check-in time is 3:00
    12.0 FAQ0558 | Q: What is the hotel's regular starting time for check-in? A: Standard


In [3]:
def bm25_scores(question):
    return bm25.get_scores(tokenize(question))

def dense_scores(question):
    q = model.encode([question], normalize_embeddings=True)[0].astype(np.float32)
    return vectors @ q

for q in ["ينفع ادخل بدري؟", "رقم الإسعاف 123", "What time is breakfast?"]:
    scores = dense_scores(q)
    print(q)
    for i in np.argsort(-scores)[:3]:
        print("   ", round(float(scores[i]), 2), chunks[i]["chunk_id"], "|", chunks[i]["text"][:70].replace("\n", " "))

ينفع ادخل بدري؟
    0.67 FAQ2561 | Q: ينفع أدخل بدري قبل الميعاد؟ A: الدخول الرسمي الساعة 3 العصر. من 12 
    0.64 FAQ2562 | Q: هوصل الصبح، أستنى لحد 3 ولا أدخل؟ A: الدخول الرسمي الساعة 3 العصر. 
    0.63 FAQ2563 | Q: ممكن early check-in؟ A: الدخول الرسمي الساعة 3 العصر. من 12 الضهر ت
رقم الإسعاف 123
    0.67 FAQ3978 | Q: ماذا أفعل في حالة طارئة أثناء التحدث مع المساعد؟ A: اطلب المساعدة ف
    0.67 FAQ3981 | Q: في حالة طوارئ أعمل إيه؟ A: اطلب المساعدة على طول: الإسعاف 123، والش
    0.63 FAQ3977 | Q: What should I do in an emergency while I am talking to the assistan
What time is breakfast?
    0.72 FAQ3391 | Q: I am not clear on the breakfast service — could you help? A: Breakf
    0.72 FAQ2078 | Q: I am trying to understand the restaurant and breakfast details — wh
    0.71 FAQ3390 | Q: What should I know about the breakfast service? A: Breakfast is ser


In [4]:
RRF_K = 60

def hybrid_rank(question, n=20):
    fused = {}
    for scores in [bm25_scores(question), dense_scores(question)]:
        order = np.argsort(-scores)[:100]
        rank = 0
        for i in order:
            if scores[i] <= 0:
                continue
            rank += 1
            fused[i] = fused.get(i, 0) + 1 / (RRF_K + rank)
    best = sorted(fused, key=fused.get, reverse=True)[:n]
    return best, fused

for q in ["ينفع ادخل بدري؟", "رقم الإسعاف 123", "What time is breakfast?", "Can I bring my dog?"]:
    best, fused = hybrid_rank(q)
    print(q)
    for i in best[:5]:
        print("   ", round(fused[i], 4), chunks[i]["chunk_id"], "|", chunks[i]["text"][:70].replace("\n", " "))

ينفع ادخل بدري؟
    0.0328 FAQ2561 | Q: ينفع أدخل بدري قبل الميعاد؟ A: الدخول الرسمي الساعة 3 العصر. من 12 
    0.0318 FAQ2562 | Q: هوصل الصبح، أستنى لحد 3 ولا أدخل؟ A: الدخول الرسمي الساعة 3 العصر. 
    0.031 FAQ3751 | Q: في خصم لو حجزت بدري؟ A: عندنا عروض موسمية في الأوقات الهادية، وخصم 
    0.0306 FAQ3609 | Q: من غير بطاقة ينفع أدخل؟ A: لازم كل الكبار يقدموا إثبات شخصية ساري و
    0.0303 FAQ3585 | Q: لازم أستنى لحد الساعة كام عشان أدخل؟ A: الدخول من الساعة 3 العصر.
رقم الإسعاف 123
    0.0325 FAQ3981 | Q: في حالة طوارئ أعمل إيه؟ A: اطلب المساعدة على طول: الإسعاف 123، والش
    0.0325 FAQ3978 | Q: ماذا أفعل في حالة طارئة أثناء التحدث مع المساعد؟ A: اطلب المساعدة ف
    0.0299 contact_information::06 | CONTACT INFORMATION - 7. Important Information For life-threatening em
    0.0289 FAQ3980 | Q: Что делать в экстренной ситуации во время разговора с помощником? A
    0.0279 FAQ3977 | Q: What should I do in an emergency while I am talking to the assistan
What time is breakfast?
    0.0304 

In [6]:
def answer_keys(chunk):
    keys = []
    if chunk["fact_id"]:
        keys.append(chunk["fact_id"])
    text = chunk["text"]
    if "\nA: " in text:
        text = text.split("\nA: ", 1)[1]
    keys.append(normalize(text))
    return keys

def search(question, top_k=5):
    try:
        if not question or not question.strip():
            return []
        best, fused = hybrid_rank(question, n=20)
        results = []
        seen = set()
        for i in best:
            keys = answer_keys(chunks[i])
            if any(k in seen for k in keys):
                continue
            seen.update(keys)
            results.append({
                "text": chunks[i]["text"],
                "source": chunks[i]["source"],
                "score": round(float(fused[i]), 4),
            })
            if len(results) == top_k:
                break
        return results
    except Exception:
        return []

for q in ["ينفع ادخل بدري؟", "What time is breakfast?"]:
    print(q)
    for r in search(q):
        print("   ", r["score"], r["source"], "|", r["text"][:60].replace("\n", " "))

print(search(""))

ينفع ادخل بدري؟
    0.0328 check_in_check_out.txt | Q: ينفع أدخل بدري قبل الميعاد؟ A: الدخول الرسمي الساعة 3 الع
    0.031 discounts_and_promotions.txt | Q: في خصم لو حجزت بدري؟ A: عندنا عروض موسمية في الأوقات الها
    0.0306 identification_policy.txt | Q: من غير بطاقة ينفع أدخل؟ A: لازم كل الكبار يقدموا إثبات شخ
    0.0303 check_in_check_out.txt | Q: لازم أستنى لحد الساعة كام عشان أدخل؟ A: الدخول من الساعة 
    0.029 check_in_check_out.txt | Q: أقدر أدخل الأوضة الساعة كام؟ A: الدخول من 3 العصر.
What time is breakfast?
    0.0304 hotel_facilities.txt | Q: I am trying to understand the restaurant and breakfast de
    0.03 breakfast.txt | Q: What should I know about the breakfast service? A: Breakf
    0.0251 breakfast.txt | BREAKFAST - 1. Overview Daily breakfast buffet in the Main R
    0.0251 restaurant.txt | Q: What is the hotel policy on the hotel restaurant? A: The 
    0.0246 room_service.txt | ROOM SERVICE - 1. Availability Room service operates 24 hour
[]


In [1]:
import sys
import csv
import time
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s
from my_rag.config import FAQ_FILE

tests = []
with open(FAQ_FILE, encoding="utf-8") as f:
    for row in csv.DictReader(f):
        if row["split"] == "test" and row["requires_tool"] == "false" and row["source_doc"] != "out_of_scope_policy":
            tests.append(row)
print(len(tests), "test questions")

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 21:05:38.556000 54452 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 47787.54it/s]


565 test questions


In [2]:
def evaluate(weight):
    s.BM25_WEIGHT = weight
    hits = {}
    totals = {}
    times = []
    for row in tests:
        start = time.time()
        results = s.search(row["question"])
        times.append(time.time() - start)
        sources = []
        for r in results:
            sources.append(r["source"])
        lang = row["language"]
        totals[lang] = totals.get(lang, 0) + 1
        if row["source_doc"] in sources:
            hits[lang] = hits.get(lang, 0) + 1
    print("BM25 weight:", weight)
    for lang in totals:
        print("   ", lang, round(100 * hits.get(lang, 0) / totals[lang]), "%")
    print("    ALL", round(100 * sum(hits.values()) / sum(totals.values())), "%")
    print("    time: avg", round(sum(times) / len(times), 3), "s | max", round(max(times), 3), "s")

for w in [1.0, 0.5, 0.0]:
    evaluate(w)

BM25 weight: 1.0
    English 89 %
    Russian 74 %
    French 79 %
    Arabic 75 %
    Arabic-Egyptian 84 %
    ALL 79 %
    time: avg 0.042 s | max 2.67 s
BM25 weight: 0.5
    English 95 %
    Russian 84 %
    French 91 %
    Arabic 81 %
    Arabic-Egyptian 84 %
    ALL 88 %
    time: avg 0.03 s | max 0.257 s
BM25 weight: 0.0
    English 100 %
    Russian 97 %
    French 99 %
    Arabic 97 %
    Arabic-Egyptian 87 %
    ALL 98 %
    time: avg 0.03 s | max 0.16 s


In [3]:
from rank_bm25 import BM25Okapi

AR = "عايز عاوز عايزة عاوزة أعمل إيه إزاي ممكن ينفع هل في من على إلى عن أنا انت إحنا ده دي دا كده بس و يا لو مع اللي هو هي عندكم فيه ما ماذا كيف متى أين هذا هذه لي ليا او"
EN = "what is are the a an can could would i my me you your do does did tell about please how when where which there in on at of to for and or it this that with be have has we our us during stay hotel nile pearl will should know like want need am if any get help"
STOPWORDS = set(s.tokenize(AR + " " + EN))

def tokenize2(text):
    words = []
    for w in s.tokenize(text):
        if w not in STOPWORDS:
            words.append(w)
    return words

print(tokenize2("عايز أعمل مساج"), tokenize2("Can I bring my dog?"))

all_corpus = []
for c in s._chunks:
    all_corpus.append(tokenize2(c["text"]))
all_bm25 = BM25Okapi(all_corpus, k1=1.5, b=0.75)

def bm25_all_stop(question):
    return all_bm25.get_scores(tokenize2(question))

kb_idx = []
for i, c in enumerate(s._chunks):
    if c["kind"] == "kb":
        kb_idx.append(i)
kb_corpus = []
for i in kb_idx:
    kb_corpus.append(tokenize2(s._chunks[i]["text"]))
kb_bm25 = BM25Okapi(kb_corpus, k1=1.5, b=0.75)

def bm25_kb_stop(question):
    full = np.zeros(len(s._chunks))
    part = kb_bm25.get_scores(tokenize2(question))
    for j, i in enumerate(kb_idx):
        full[i] = part[j]
    return full

original = s._bm25_scores

print("=== Solution 1: stopwords, all chunks")
s._bm25_scores = bm25_all_stop
for w in [0.5, 0.2]:
    evaluate(w)

print("=== Solution 1+2: stopwords, knowledge files only")
s._bm25_scores = bm25_kb_stop
for w in [1.0, 0.5]:
    evaluate(w)

s._bm25_scores = original

['مساج'] ['bring', 'dog']
=== Solution 1: stopwords, all chunks
BM25 weight: 0.5
    English 91 %
    Russian 84 %
    French 91 %
    Arabic 84 %
    Arabic-Egyptian 82 %
    ALL 87 %
    time: avg 0.029 s | max 1.014 s
BM25 weight: 0.2
    English 99 %
    Russian 91 %
    French 95 %
    Arabic 90 %
    Arabic-Egyptian 87 %
    ALL 93 %
    time: avg 0.027 s | max 0.06 s
=== Solution 1+2: stopwords, knowledge files only
BM25 weight: 1.0
    English 92 %
    Russian 97 %
    French 98 %
    Arabic 97 %
    Arabic-Egyptian 87 %
    ALL 96 %
    time: avg 0.025 s | max 0.073 s
BM25 weight: 0.5
    English 97 %
    Russian 97 %
    French 99 %
    Arabic 97 %
    Arabic-Egyptian 87 %
    ALL 97 %
    time: avg 0.025 s | max 0.054 s


In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "knowledge_base").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from my_rag import search as s
from my_rag.search import search
print("BM25_WEIGHT =", s.BM25_WEIGHT)

questions = [
    "عايز late check-out، بكام؟",
    "بتراجعوا الحجز إزاي قبل الإلغاء؟",
    "النزيل اللي جنبي بيدخن في أوضته والريحة داخلة عندي",
    "عندكم إيه في منيو الروم سيرفس؟",
    "الفندق فيه كام أوضة؟",
    "عايز أعمل مساج",
    "Can I bring my dog?",
    "أبغى أعرف متى الفطور؟",
    "وش أوقات المسبح؟",
    "What's the weather tomorrow?",
]

for q in questions:
    start = time.time()
    results = search(q)
    t = time.time() - start
    print(q, f"({round(t, 3)} s)")
    for r in results[:3]:
        print("   ", r["score"], r["source"], "|", r["text"][:60].replace("\n", " "))

/Users/sondosahmed/projects/HotelAI-Intelligent-Agent_Final_Data/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 21:28:43.872000 55453 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 43789.83it/s]


BM25_WEIGHT = 0.0
عايز late check-out، بكام؟ (0.131 s)
    0.7143 check_in_check_out.txt | Q: If I need to leave later than the standard time, can I re
    0.7094 check_in_check_out.txt | Q: Could I stay in the room beyond the regular departure tim
    0.7041 check_in_check_out.txt | Q: I need to stay in my room longer on departure day. Can la
بتراجعوا الحجز إزاي قبل الإلغاء؟ (0.075 s)
    0.6771 no_show_policy.txt | Q: أحسن ألغي ولا ماأجيش وخلاص؟ A: الإلغاء إنك تلغي الحجز قبل
    0.6685 booking_modification.txt | Q: أسافر إلى القاهرة للعمل وسأقيم في Nile Pearl Hotel، فما ا
    0.6642 cancellation_policy.txt | Q: لو لغيت هتخصموا مني كام؟ A: في السعر المرن الإلغاء ببلاش 
النزيل اللي جنبي بيدخن في أوضته والريحة داخلة عندي (0.075 s)
    0.5009 booking_modification.txt | Q: هل يمكنك توضيح البيانات المطلوبة عند طلب التعديل لي؟ A: ي
    0.4945 booking_modification.txt | Q: نسيت رقم الحجز خالص، أعمل إيه؟ A: كلّم خدمة العملاء وقوله
    0.4932 room_types.txt | Q: الـ Suite فيها إيه؟ A: الجناح ف

In [2]:
print(search(""))
print(search("   "))
print(search(None))
print(search(12345))
print(len(search("x" * 10000)))

r = search("What time is breakfast?")
print(type(r), len(r), sorted(r[0].keys()))
print(len(search("What time is breakfast?", top_k=3)))


[]
[]
[]
[]
5
<class 'list'> 5 ['score', 'source', 'text']
3
